In [ ]:
!pip install kagglehub pandas

In [ ]:

# Import required libraries
import kagglehub
import pandas as pd
import os
import re
from email.parser import Parser
from email.utils import parseaddr

# Step 1: Download dataset from Kaggle
path = kagglehub.dataset_download(
    "rtatman/fraudulent-email-corpus"
)

print("Dataset downloaded successfully!")
print("Dataset location:", path)

# Step 2: Locate the email dataset
file_path = None

for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower() == "fradulent_emails.txt":
            file_path = os.path.join(root, file)
            break

if file_path is None:
    raise FileNotFoundError(
        "fradulent_emails.txt not found in downloaded dataset"
    )

# Step 3: Read the dataset
with open(file_path, "r", encoding="latin1") as f:
    data = f.read()

# Step 4: Split individual emails
emails = [
    "From r" + part
    for part in data.split("From r")[1:]
]

print("Total emails loaded:", len(emails))

# Step 5: Extract email headers
records = []

def get_domain(address):
    address = parseaddr(address)[1]
    if "@" in address:
        return address.split("@")[-1].lower()
    return ""

for email_text in emails:
    try:
        msg = Parser().parsestr(email_text)

        sender = msg.get("From", "")
        reply_to = msg.get("Reply-To", "")
        return_path = msg.get("Return-Path", "")

        sender_domain = get_domain(sender)
        reply_domain = get_domain(reply_to)
        return_domain = get_domain(return_path)

        # Detect suspicious header mismatches
        flags = []

        if sender_domain and reply_domain:
            if sender_domain != reply_domain:
                flags.append("From-Reply-To mismatch")

        if sender_domain and return_domain:
            if sender_domain != return_domain:
                flags.append("From-Return-Path mismatch")

        if not sender_domain:
            flags.append("Missing or invalid sender")

        if flags:
            status = "Suspicious - Review"
        else:
            status = "No mismatch detected"

        records.append({
            "Sender": sender,
            "Reply-To": reply_to,
            "Return-Path": return_path,
            "Sender Domain": sender_domain,
            "Reply Domain": reply_domain,
            "Return Domain": return_domain,
            "Spoofing Indicators": ", ".join(flags),
            "Status": status
        })

    except Exception:
        continue

# Step 6: Create DataFrame
df = pd.DataFrame(records)

# Step 7: Display results
print("\nEMAIL SPOOFING ANALYSIS")
print(df.head(10).to_string(index=False))

# Step 8: Display summary
print("\nANALYSIS SUMMARY")
print("Total emails analyzed:", len(df))

print("\nDetection results:")
print(df["Status"].value_counts())

# Step 9: Save results
df.to_csv("email_spoofing_results.csv", index=False)

print("\nResults saved to email_spoofing_results.csv")


100%|██████████| 5.52M/5.52M [00:01<00:00, 4.72MB/s]

Extracting files...


Dataset downloaded successfully!
Dataset location: /root/.cache/kagglehub/datasets/rtatman/fraudulent-email-corpus/versions/1
Total emails loaded: 3977

EMAIL SPOOFING ANALYSIS
                                           Sender                      Reply-To                     Return-Path  Sender Domain Reply Domain  Return Domain       Spoofing Indicators               Status
 "MR. JAMES NGOLA." <james_ngola2002@maktoob.com>   james_ngola2002@maktoob.com   <james_ngola2002@maktoob.com>    maktoob.com  maktoob.com    maktoob.com                           No mismatch detected
 "Mr. Ben Suleman" <bensul2004nng@spinfinder.com>                                <bensul2004nng@spinfinder.com> spinfinder.com              spinfinder.com                           No mismatch detected
      "PRINCE OBONG ELEME" <obong_715@epatra.com>          obong_715@epatra.com          <obong_715@epatra.com>     epatra.com   epatra.com     epatra.com                           No mismatch detected
      "PRINCE O